Model Training & Evaluation


In [ ]:
import pandas as pd
import numpy as np, joblib
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
df=pd.read_csv("/content/Nassau_Candy_Cleaned (1).csv")

In [ ]:
features=["Ship Mode","Region","Division","Factory","Sales","Units","Cost","Gross Profit","Profit Margin","Cost Percentage","Unit Price"]
target="Lead_Time"
df=df.dropna(subset=[target]).copy()
X=df[features]; y=df[target]
cat=["Ship Mode","Region","Division","Factory"]; num=[c for c in features if c not in cat]
pre=ColumnTransformer([("cat",OneHotEncoder(handle_unknown="ignore"),cat),
                       ("num",SimpleImputer(strategy="median"),num)])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)


In [ ]:
models={
"Linear Regression":LinearRegression(),
"Random Forest":RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1),
"Gradient Boosting":GradientBoostingRegressor(random_state=42)
}
results=[]; fitted={}
for name,est in models.items():
    pipe=Pipeline([("pre",pre),("model",est)])
    pipe.fit(X_train,y_train); pred=pipe.predict(X_test)
    results.append([name,mean_absolute_error(y_test,pred),mean_squared_error(y_test,pred)**.5,r2_score(y_test,pred)])
    fitted[name]=pipe
comparison=pd.DataFrame(results,columns=["Model","MAE","RMSE","R2"]).sort_values("RMSE")
comparison

,Model,MAE,RMSE,R2
0,Linear Regression,253.847361,309.105909,-0.007895
2,Gradient Boosting,257.089315,313.554004,-0.037111
1,Random Forest,269.290541,333.596161,-0.173931


In [ ]:
import os

best_name=comparison.iloc[0]["Model"]
best_model=fitted[best_name]

# Create directories if they don't exist
os.makedirs("../models", exist_ok=True)
os.makedirs("../reports", exist_ok=True)

joblib.dump(best_model,"/content/best_model_random_forest.pkl")
comparison.to_csv("../reports/model_comparison.csv",index=False)
print("Best model:",best_name)

Best model: Linear Regression


In [ ]:
import os
import joblib

project_path = "/content/drive/MyDrive/Your Project"

os.makedirs(f"{project_path}/models", exist_ok=True)
os.makedirs(f"{project_path}/reports", exist_ok=True)

joblib.dump(best_model, f"/content/best_model_random_forest.pkl")

comparison.to_csv(
    f"/content/model_comparison.csv",
    index=False
)

print("Best model:", best_name)
print("Model saved successfully!")

Best model: Linear Regression
Model saved successfully!


In [ ]:
# ============================================================
# IMPROVED FEATURES FOR LEAD TIME PREDICTION
# ============================================================

import pandas as pd
import numpy as np

# Make a copy so the original dataframe is not modified
model_df = df.copy()

# Convert Order Date to datetime
model_df["Order Date"] = pd.to_datetime(
    model_df["Order Date"],
    errors="coerce"
)

# Create useful time features
model_df["Order Year"] = model_df["Order Date"].dt.year
model_df["Order Month"] = model_df["Order Date"].dt.month
model_df["Order Quarter"] = model_df["Order Date"].dt.quarter
model_df["Order Day of Week"] = model_df["Order Date"].dt.dayofweek

print("New date features created.")

print(
    model_df[
        [
            "Order Date",
            "Order Year",
            "Order Month",
            "Order Quarter",
            "Order Day of Week"
        ]
    ].head()
)

New date features created.
  Order Date  Order Year  Order Month  Order Quarter  Order Day of Week
0 2024-01-03        2024            1              1                  2
1 2024-01-04        2024            1              1                  3
2 2024-01-04        2024            1              1                  3
3 2024-01-04        2024            1              1                  3
4 2024-01-05        2024            1              1                  4


In [ ]:
# ============================================================
# IMPROVED MODEL TRAINING
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

import numpy as np
import pandas as pd
import joblib
from pathlib import Path


# ------------------------------------------------------------
# FEATURES
# ------------------------------------------------------------

categorical_features = [
    "Product ID",
    "Product Name",
    "Division",
    "Region",
    "State/Province",
    "Ship Mode",
    "Factory"
]

numerical_features = [
    "Sales",
    "Units",
    "Cost",
    "Gross Profit",
    "Profit Margin",
    "Cost Percentage",
    "Unit Price",
    "Order Year",
    "Order Month",
    "Order Quarter",
    "Order Day of Week"
]

target = "Lead_Time"


# ------------------------------------------------------------
# KEEP ONLY EXISTING COLUMNS
# ------------------------------------------------------------

categorical_features = [
    col for col in categorical_features
    if col in model_df.columns
]

numerical_features = [
    col for col in numerical_features
    if col in model_df.columns
]

all_features = categorical_features + numerical_features

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)


# ------------------------------------------------------------
# REMOVE ROWS WITH MISSING TARGET
# ------------------------------------------------------------

training_df = model_df[
    all_features + [target]
].copy()

training_df = training_df.dropna(
    subset=[target]
).reset_index(drop=True)


X = training_df[all_features]
y = training_df[target]


# ------------------------------------------------------------
# TRAIN TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# ------------------------------------------------------------
# PREPROCESSING
# ------------------------------------------------------------

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

numerical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_transformer,
            categorical_features
        ),
        (
            "numerical",
            numerical_transformer,
            numerical_features
        )
    ]
)


# ------------------------------------------------------------
# MODELS
# ------------------------------------------------------------

models = {

    "Linear Regression":
        LinearRegression(),

    "Random Forest":
        RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ),

    "Gradient Boosting":
        GradientBoostingRegressor(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        )
}


# ------------------------------------------------------------
# TRAIN AND EVALUATE
# ------------------------------------------------------------

results = []

trained_models = {}

for name, regressor in models.items():

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "model",
                regressor
            )
        ]
    )

    print("\nTraining:", name)

    pipeline.fit(
        X_train,
        y_train
    )

    predictions = pipeline.predict(
        X_test
    )

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

    trained_models[name] = pipeline


# ------------------------------------------------------------
# MODEL COMPARISON
# ------------------------------------------------------------

model_results = pd.DataFrame(
    results
).sort_values(
    "RMSE",
    ascending=True
).reset_index(drop=True)

print("\n======================================")
print("IMPROVED MODEL COMPARISON")
print("======================================")

display(model_results)

Categorical features:
['Product ID', 'Product Name', 'Division', 'Region', 'State/Province', 'Ship Mode', 'Factory']

Numerical features:
['Sales', 'Units', 'Cost', 'Gross Profit', 'Profit Margin', 'Cost Percentage', 'Unit Price', 'Order Year', 'Order Month', 'Order Quarter', 'Order Day of Week']

Training: Linear Regression

Training: Random Forest

Training: Gradient Boosting

IMPROVED MODEL COMPARISON


,Model,MAE,RMSE,R2
0,Random Forest,138.591989,172.148231,0.687388
1,Gradient Boosting,168.658759,195.796179,0.595602
2,Linear Regression,183.323031,211.913175,0.526286


In [ ]:
# Show variables currently available
print([
    name for name in globals()
    if "model" in name.lower() or "rf" in name.lower() or "forest" in name.lower()
])

['RandomForestRegressor', 'models', 'best_model', 'model_df', 'trained_models', 'model_results']


In [ ]:
print(type(trained_models))
print(trained_models.keys())

<class 'dict'>
dict_keys(['Linear Regression', 'Random Forest', 'Gradient Boosting'])


In [ ]:
for name, model in trained_models.items():
    print("\nMODEL:", name)
    print(model)


MODEL: Linear Regression
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Product ID', 'Product Name',
                                                   'Division', 'Region',
                                                   'State/Province',
                                                   'Ship Mode', 'Factory']),
                                                 ('numerical',
                                                  Pipeline(steps=[('imputer',
                                                            

In [ ]:
# ============================================================
# SAVE IMPROVED RANDOM FOREST MODEL
# ============================================================

import joblib
import os

# Select Random Forest explicitly
best_model = trained_models["Random Forest"]

print("Model being saved:")
print(best_model)

# Save to /content
model_path = "/content/best_model_random_forest.pkl"

joblib.dump(
    best_model,
    model_path
)

print("\nModel saved successfully!")
print("Path:", model_path)
print("File size:", os.path.getsize(model_path), "bytes")

Model being saved:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Product ID', 'Product Name',
                                                   'Division', 'Region',
                                                   'State/Province',
                                                   'Ship Mode', 'Factory']),
                                                 ('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   

In [ ]:
# ============================================================
# VERIFY RANDOM FOREST MODEL
# ============================================================

import joblib

model_test = joblib.load(
    "/content/best_model_random_forest.pkl"
)

print("Model loaded successfully!")
print(model_test)

Model loaded successfully!
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Product ID', 'Product Name',
                                                   'Division', 'Region',
                                                   'State/Province',
                                                   'Ship Mode', 'Factory']),
                                                 ('numerical',
                                                  Pipeline(steps=[('imputer',
                                                           

In [ ]:
import joblib

model = joblib.load(
    "/content/best_model_random_forest.pkl"
)

print("Recommendation model loaded successfully!")
print(model)

Recommendation model loaded successfully!
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Product ID', 'Product Name',
                                                   'Division', 'Region',
                                                   'State/Province',
                                                   'Ship Mode', 'Factory']),
                                                 ('numerical',
                                                  Pipeline(steps=[('imputer',
                                            